# 01 · Exploração dos dados sintéticos

**Gazeta Meridiano** é uma empresa fictícia; os dados são 100% sintéticos (seed fixa). Este notebook explora o conjunto antes da análise formal: atributos, entradas, censura à direita e as primeiras curvas. A análise A–H está no notebook 02.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
raiz = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(raiz / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from sobrevivencia import config as C, simulate as S, km as K, cohorts as H, cox as X
from sobrevivencia import value as V, pitfalls as P, competing as G, figures as F

pd.options.display.float_format = "{:,.3f}".format
pd.options.display.max_columns = 30
F._estilo()
df, lat = S.simular()
gt = S.construir_ground_truth(df, lat)
import tempfile
FIG = Path(tempfile.mkdtemp(prefix='figuras_nb_'))
print(f"{len(df):,} assinantes | seed {C.SEED} | corte {C.CORTE}")

12,000 assinantes | seed 42 | corte 2026-10-31


## Os dados

Uma linha por assinante. `duracao_meses` é o tempo observado e `evento` vale 1 se cancelou e 0 se foi censurado no corte.

In [2]:
df.head(8)

,id_assinante,data_inicio,canal,plano,oferta_entrada,dispositivo,engajamento_pre,safra,seguimento_max_meses,duracao_meses,evento,causa,data_cancelamento
0,A000001,2024-11-01,busca_paga,mensal,nenhuma,desktop,60.100,S1,23.984,0.619,1,involuntario,2024-11-19
1,A000002,2024-11-01,organico,combo,nenhuma,tablet,31.500,S1,23.984,23.984,0,,NaT
2,A000003,2024-11-01,midia_paga,mensal,nenhuma,desktop,40.900,S1,23.984,14.063,1,voluntario,2026-01-03
3,A000004,2024-11-01,organico,mensal,nenhuma,mobile,45.100,S1,23.984,6.477,1,voluntario,2025-05-17
4,A000005,2024-11-01,email_crm,combo,nenhuma,mobile,73.000,S1,23.984,23.984,0,,NaT
5,A000006,2024-11-01,busca_paga,mensal,nenhuma,mobile,44.600,S1,23.984,23.984,0,,NaT
6,A000007,2024-11-01,indicacao,combo,nenhuma,mobile,63.700,S1,23.984,23.984,0,,NaT
7,A000008,2024-11-01,organico,anual,nenhuma,tablet,54.500,S1,23.984,23.984,0,,NaT


In [3]:
df.describe(include='all').T[['count', 'unique', 'top', 'mean', 'min', 'max']]

,count,unique,top,mean,min,max
id_assinante,12000,12000,A000001,NaN,NaN,NaN
data_inicio,12000,NaN,NaN,2025-11-14 04:38:24,2024-11-01 00:00:00,2026-10-31 00:00:00
canal,12000,6,organico,NaN,NaN,NaN
plano,12000,4,mensal,NaN,NaN,NaN
oferta_entrada,12000,3,nenhuma,NaN,NaN,NaN
dispositivo,12000,3,mobile,NaN,NaN,NaN
engajamento_pre,"12,000.000",NaN,NaN,51.299,0.000,100.000
safra,12000,8,S5,NaN,NaN,NaN
seguimento_max_meses,"12,000.000",NaN,NaN,11.558,0.033,23.984
duracao_meses,"12,000.000",NaN,NaN,8.815,0.010,23.984


## Entradas e campanhas

A intensidade de entrada tem picos nas campanhas (Black Friday 2024 e 2025, Passe Copa e Eleições 2026).

In [4]:
entradas = df.groupby(df['data_inicio'].dt.to_period('M')).size()
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.bar(entradas.index.astype(str), entradas.values, color=C.COR['meridiano'])
ax.set_title('Novos assinantes por mês de entrada'); ax.tick_params(axis='x', rotation=70)
plt.show()
entradas.describe()

count      24.000
mean      500.000
std       178.660
min       356.000
25%       397.750
50%       423.500
75%       567.500
max     1,135.000
dtype: float64

In [5]:
mix = pd.crosstab(df['safra'], df['canal'], normalize='index')
mix.style.format('{:.1%}').background_gradient(cmap='Blues', axis=None)

canal,busca_paga,email_crm,indicacao,midia_paga,organico,social
safra,,,,,,
S1,18.3%,13.6%,6.7%,24.5%,25.7%,11.3%
S2,18.4%,12.4%,8.8%,19.9%,27.4%,13.1%
S3,20.6%,12.4%,7.4%,18.3%,29.3%,11.9%
S4,17.3%,11.1%,7.4%,21.7%,29.8%,12.8%
S5,18.4%,13.8%,7.2%,25.5%,24.5%,10.6%
S6,18.3%,12.6%,7.0%,20.4%,30.9%,10.8%
S7,16.6%,10.7%,7.9%,21.4%,29.3%,14.2%
S8,14.5%,10.6%,7.0%,16.3%,36.4%,15.1%


In [6]:
pd.concat({c: df[c].value_counts(normalize=True) for c in ['plano', 'oferta_entrada', 'dispositivo']}, axis=1)

,plano,oferta_entrada,dispositivo
mensal,0.455,NaN,NaN
combo,0.200,NaN,NaN
estudante,0.183,NaN,NaN
anual,0.162,NaN,NaN
nenhuma,NaN,0.871,NaN
black_friday,NaN,0.093,NaN
passe_copa,NaN,0.035,NaN
mobile,NaN,NaN,0.612
desktop,NaN,NaN,0.308
tablet,NaN,NaN,0.081


## Censura à direita

O corte é fixo, então quem entrou tarde tem pouco tempo de observação. A figura mostra a vida de 40 assinantes: `x` = cancelou, círculo vazado = censurado (ainda ativo no corte).

In [7]:
amostra = df.sample(40, random_state=C.SEED).sort_values('data_inicio')
fig, ax = plt.subplots(figsize=(10, 5))
for i, (_, r) in enumerate(amostra.iterrows()):
    ax.hlines(i, 0, r['duracao_meses'], color=C.COR['meridiano'], lw=1.4)
    ax.plot(r['duracao_meses'], i, 'x' if r['evento'] else 'o', color=C.COR['vermelho'] if r['evento'] else C.COR['cinza'],
            mfc='none' if not r['evento'] else None)
ax.set_xlabel('meses desde a primeira cobrança'); ax.set_yticks([]); ax.set_title('Linhas de vida (x = cancelou, o = censurado)')
plt.show()

In [8]:
cens = df.groupby('safra').agg(n=('evento', 'size'), cancelaram=('evento', 'sum'), seguimento_min=('seguimento_max_meses', 'min'),
                               seguimento_max=('seguimento_max_meses', 'max'))
cens['pct_censurado'] = 1 - cens['cancelaram'] / cens['n']
cens

,n,cancelaram,seguimento_min,seguimento_max,pct_censurado
safra,,,,,
S1,1679,918,20.994,23.984,0.453
S2,1153,549,18.070,20.961,0.524
S3,1265,575,15.047,18.037,0.545
S4,1199,478,12.025,15.014,0.601
S5,2019,699,9.002,11.992,0.654
S6,1215,324,6.078,8.969,0.733
S7,1689,304,3.055,6.045,0.820
S8,1781,98,0.033,3.023,0.945


## Primeiras curvas

Kaplan-Meier de toda a base e por plano. O plano anual quase não cai até o mês 12 e depois despenca na renovação.

In [9]:
fig, ax = plt.subplots(figsize=(9, 4.6))
F._curva_km(ax, df, C.COR['tinta'], 'Todos')
for p in C.PLANOS:
    F._curva_km(ax, df[df['plano'] == p], C.COR_PLANO[p], C.PLANOS_ROTULO[p], ic=False)
ax.set_xlim(0, 24); ax.set_ylim(0, 1.02); ax.legend(); ax.set_xlabel('meses'); ax.set_title('Kaplan-Meier: base inteira e por plano')
plt.show()

## Risco empírico por mês de vida

O risco mensal (eventos ÷ tempo sob risco) mostra o **penhasco** da promoção: quem entrou pela Black Friday cancela muito mais no mês 3–4, o primeiro a preço cheio.

In [10]:
na = df[df['plano'] != 'anual']
bf, ref = na[na['oferta_entrada'] == 'black_friday'], na[na['oferta_entrada'] == 'nenhuma']
tab = pd.DataFrame({'sem_promocao': [r['taxa'] for r in K.hazard_por_idade(ref['duracao_meses'], ref['evento'], 12)],
                    'black_friday': [r['taxa'] for r in K.hazard_por_idade(bf['duracao_meses'], bf['evento'], 12)]})
tab.index.name = 'mês de vida'
ax = tab.plot(drawstyle='steps-post', figsize=(9, 3.8), color=[C.COR['meridiano'], C.COR['vermelho']])
ax.set_ylabel('risco mensal'); ax.set_title('Risco mensal de cancelar por mês de vida'); plt.show()
tab.round(4)

,sem_promocao,black_friday
mês de vida,,
0,0.054,0.046
1,0.046,0.053
2,0.042,0.038
3,0.037,0.110
4,0.037,0.065
5,0.037,0.047
6,0.034,0.038
7,0.038,0.043
8,0.035,0.050


## O que o KM recupera da verdade plantada

Como o gerador é conhecido, dá para comparar o KM por canal com a sobrevivência verdadeira.

In [11]:
linhas = []
for c in C.CANAIS:
    sub = df[df['canal'] == c]
    for h in (12, 18):
        p = K.km_ponto(sub['duracao_meses'], sub['evento'], h)
        v = gt['sobrevivencia_verdade']['por_canal'][c][f'm{h}']['todos']
        linhas.append({'canal': C.CANAIS_ROTULO[c], 'h': f'M{h}', 'KM': p['valor'], 'IC_lo': p['lo'], 'IC_hi': p['hi'], 'verdade': v,
                       'cobre': p['lo'] <= v <= p['hi']})
pd.DataFrame(linhas)

,canal,h,KM,IC_lo,IC_hi,verdade,cobre
0,Orgânico,M12,0.710,0.691,0.729,0.696,True
1,Orgânico,M18,0.568,0.544,0.592,0.556,True
2,Busca paga,M12,0.647,0.623,0.670,0.644,True
3,Busca paga,M18,0.486,0.455,0.516,0.488,True
4,Mídia paga (Meta),M12,0.559,0.536,0.582,0.530,False
5,Mídia paga (Meta),M18,0.371,0.344,0.398,0.354,True
6,Social,M12,0.569,0.537,0.600,0.599,True
7,Social,M18,0.435,0.398,0.471,0.449,True
8,Indicação,M12,0.811,0.777,0.839,0.800,True
9,Indicação,M18,0.687,0.642,0.728,0.683,True


## Próximo passo

O notebook **02 · Análise** aplica o desenho A–H: ingênuo × KM por safra, log-rank, Cox com a violação mostrada e corrigida, valor por canal, armadilhas e riscos competitivos.